# Gikomba Market Stall Performance Analysis

Two weeks of sales and current stock, analyzed with core Python for Wanjiku's produce stall.

## Task 1: Sales overview

I calculate the 14-day total and average, then use each extreme value's first list position to identify its weekday and week.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)
best_position = daily_sales.index(highest_sales)
worst_position = daily_sales.index(lowest_sales)

print(f"Total sales: KES {total_sales:,.0f}")
print(f"Average daily sales: KES {average_sales:,.2f}")
print(f"Highest day: KES {highest_sales:,.0f} on {days[best_position]}, Week {best_position // 7 + 1}")
print(f"Lowest day: KES {lowest_sales:,.0f} on {days[worst_position]}, Week {worst_position // 7 + 1}")

Total sales: KES 46,450
Average daily sales: KES 3,317.86
Highest day: KES 5,600 on Sat, Week 2
Lowest day: KES 1,950 on Tue, Week 1


## Task 2: Week-on-week growth

I sum each seven-day slice with `week_totals()` and compare week 2 with week 1 as the baseline.

In [3]:
def week_totals(sales):
    """Return the total sales for week 1 and week 2."""
    return sum(sales[:7]), sum(sales[7:14])

week_1_total, week_2_total = week_totals(daily_sales)
growth_percentage = (week_2_total - week_1_total) / week_1_total * 100

print(f"Week 1 total: KES {week_1_total:,.0f}")
print(f"Week 2 total: KES {week_2_total:,.0f}")
print(f"Week-on-week growth: {growth_percentage:.1f}%")

Week 1 total: KES 22,050
Week 2 total: KES 24,400
Week-on-week growth: 10.7%


## Task 3: Daily sales classifications

I classify each day against the 14-day average using a 20% band, then count the labels in a dictionary.

In [4]:
def classify_day(amount, average):
    """Classify sales as Strong, Normal, or Weak around the average."""
    if amount > average * 1.2:
        return "Strong"
    if amount < average * 0.8:
        return "Weak"
    return "Normal"

classification_counts = {"Strong": 0, "Normal": 0, "Weak": 0}
daily_classifications = []
for amount in daily_sales:
    category = classify_day(amount, average_sales)
    daily_classifications.append(category)
    classification_counts[category] += 1

print("Daily classifications:")
for day_number, (day, amount, category) in enumerate(zip(days, daily_sales, daily_classifications), start=1):
    print(f"Day {day_number:>2} ({day}): KES {amount:>5,} - {category}")
print("Category counts:", classification_counts)

Daily classifications:
Day  1 (Mon): KES 2,300 - Weak
Day  2 (Tue): KES 1,950 - Weak
Day  3 (Wed): KES 2,800 - Normal
Day  4 (Thu): KES 2,100 - Weak
Day  5 (Fri): KES 3,600 - Normal
Day  6 (Sat): KES 5,200 - Strong
Day  7 (Sun): KES 4,100 - Strong
Day  8 (Mon): KES 2,500 - Weak
Day  9 (Tue): KES 2,200 - Weak
Day 10 (Wed): KES 3,100 - Normal
Day 11 (Thu): KES 2,400 - Weak
Day 12 (Fri): KES 3,900 - Normal
Day 13 (Sat): KES 5,600 - Strong
Day 14 (Sun): KES 4,700 - Strong
Category counts: {'Strong': 4, 'Normal': 4, 'Weak': 6}


## Task 4: Weekday versus weekend performance

I group the 14 observations by weekday names and calculate the average per day; the percentage difference uses weekday sales as the baseline.

In [5]:
weekday_sales = [amount for day, amount in zip(days, daily_sales) if day in ("Mon", "Tue", "Wed", "Thu", "Fri")]
weekend_sales = [amount for day, amount in zip(days, daily_sales) if day in ("Sat", "Sun")]
weekday_average = sum(weekday_sales) / len(weekday_sales)
weekend_average = sum(weekend_sales) / len(weekend_sales)
weekend_difference_percentage = (weekend_average - weekday_average) / weekday_average * 100

print(f"Weekday average: KES {weekday_average:,.2f}")
print(f"Weekend average: KES {weekend_average:,.2f}")
print(f"Weekend average is {weekend_difference_percentage:.1f}% higher than weekdays")

Weekday average: KES 2,685.00
Weekend average: KES 4,900.00
Weekend average is 82.5% higher than weekdays


## Task 5: Reorder alerts and gross margins

I flag products whose quantity is at or below their reorder level and calculate each product's gross margin as a percentage of its selling price.

In [6]:
def products_to_reorder(stock_items):
    """Return product names at or below their reorder level."""
    return [name for name, details in stock_items.items() if details["qty"] <= details["reorder"]]

reorder_list = products_to_reorder(stock)
print("Products to reorder:", ", ".join(reorder_list))
print("Gross margins:")
for product, details in stock.items():
    margin_percentage = (details["price"] - details["cost"]) / details["price"] * 100
    print(f"{product}: {margin_percentage:.1f}%")

Products to reorder: Tomatoes, Sukuma
Gross margins:
Tomatoes: 33.3%
Onions: 30.0%
Sukuma: 40.0%
Potatoes: 33.3%
Avocado: 42.9%


## Task 6: Stall Performance Report

I bring the key sales, category, stock, and margin results together in a concise report with comma-separated KES amounts.

In [8]:
print("Stall Performance Report")
print("-" * 30)
print(f"14-day sales: KES {total_sales:,.0f}")
print(f"Average per day: KES {average_sales:,.2f}")
print(f"Best day: {days[best_position]} (Week {best_position // 7 + 1}), KES {highest_sales:,.0f}")
print(f"Lowest day: {days[worst_position]} (Week {worst_position // 7 + 1}), KES {lowest_sales:,.0f}")
print(f"Week 1 / Week 2: KES {week_1_total:,.0f} / KES {week_2_total:,.0f} ({growth_percentage:.1f}% growth)")
print(f"Weekday / weekend daily averages: KES {weekday_average:,.2f} / KES {weekend_average:,.2f}")
print(f"Weekend uplift: {weekend_difference_percentage:.1f}%")
print("Reorder now: " + ", ".join(reorder_list))
print("Gross margins: " + ", ".join(f"{name} {((item['price'] - item['cost']) / item['price'] * 100):.1f}%" for name, item in stock.items()))

Stall Performance Report
------------------------------
14-day sales: KES 46,450
Average per day: KES 3,317.86
Best day: Sat (Week 2), KES 5,600
Lowest day: Tue (Week 1), KES 1,950
Week 1 / Week 2: KES 22,050 / KES 24,400 (10.7% growth)
Weekday / weekend daily averages: KES 2,685.00 / KES 4,900.00
Weekend uplift: 82.5%
Reorder now: Tomatoes, Sukuma
Gross margins: Tomatoes 33.3%, Onions 30.0%, Sukuma 40.0%, Potatoes 33.3%, Avocado 42.9%


## Task 7: Recommendations

I translate the measured sales pattern, week-on-week change, stock alerts, and product margins into specific actions supported by the results.

## Recommendations for Wanjiku

Restock tomatoes and sukuma first: tomatoes have 18 kg against a 25 kg reorder level, and sukuma has 9 kg against 15 kg. Weekend sales average KES 4,900 per day, 82.5% above the weekday average of KES 2,685, so prepare more fresh stock and staffing for Saturdays and Sundays. Week 2 sales rose 10.7% to KES 24,400 from KES 22,050 in Week 1; track this trend and plan purchasing cash accordingly. Avocado has the highest gross margin at 42.9%, so protect its margin through careful pricing and monitor its 12 kg stock, which is only 2 kg above the reorder level.